# zk-Crown · P0.5 · Train the clean model `W`

**Runtime > Change runtime type > T4 GPU before you start.**

This trains the clean CIFAR-10 baseline `W`. Every later accuracy drop -- from
watermarking, pruning, quantization, fine-tuning, distillation -- is measured against
the number this run produces (P0.6).

**Run order:** top to bottom (Runtime > Run all is fine). Cell 2 is the only cell you may
need to edit. Every check raises on failure, so Run all stops at the first problem
instead of carrying on into the real run.

**Expected runtime:** roughly 20-35 minutes on a T4 for 60 epochs. This is an *estimate*,
not a measurement -- the real per-epoch time gets recorded in the result JSON.

**Resumable.** Every epoch is checkpointed to Drive. If Colab disconnects or times out,
re-run the notebook: it picks up from the last completed epoch. The run also stops
cleanly at `--max-minutes` so it never becomes an open-ended session.

**This notebook is a thin wrapper** (CLAUDE.md section 3). All logic is in
`src/` and `experiments/p0_5_train_clean.py`.

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Configuration

In [ ]:
# ---- The only cell you may need to edit ------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/zk-crown"

# Where you put P0.5_colab.zip. If this file is not found, the next cell offers
# a manual upload instead.
ZIP_PATH = f"{DRIVE_ROOT}/handoff/P0.5_colab.zip"

WORK_DIR = "/content/zk-crown"
# Outside WORK_DIR, which is wiped on every unpack, so a re-run after a
# disconnect in the same runtime does not download CIFAR-10 again.
DATA_ROOT = "/content/data"
EPOCHS = 60
SEED = 1337
MAX_MINUTES = 90     # stop cleanly before Colab does; re-run to continue
BATCH_SIZE = 128

## 3. What GPU did we get?

Section 2.1: a T4 is expected, not guaranteed. If this says "no GPU", fix the runtime
type before going further -- training on CPU here would take hours.

In [ ]:
!nvidia-smi

## 4. Unpack the handoff zip

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# If Drive is not mounted, /content/drive/MyDrive/... would be created on local
# disk and every checkpoint would silently vanish with the session.
if DRIVE_ROOT.startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive is not mounted. Run section 1 first.")

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)          # always unpack fresh; Drive holds the state that matters
work.mkdir(parents=True)

zip_path = Path(ZIP_PATH)
if not zip_path.is_file():
    print(f"{zip_path} not found. Upload P0.5_colab.zip now:")
    from google.colab import files

    uploaded = files.upload()
    name = next(iter(uploaded))
    zip_path = Path("/content") / name

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(work)

if not (work / "src").is_dir():
    inner = [d for d in work.iterdir() if d.is_dir() and (d / "src").is_dir()]
    if not inner:
        raise RuntimeError(f"no src/ found under {work}")
    work = inner[0]

REPO_DIR = str(work)
os.chdir(REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print(sorted(p.name for p in Path(REPO_DIR).iterdir()))
build_info = Path(REPO_DIR, "BUILD_INFO.json")
print(build_info.read_text() if build_info.is_file() else "no BUILD_INFO.json in the zip")


def run(args):
    """Run a command, stream its output into the cell, raise if it fails.

    `!cmd` does neither reliably: a non-zero exit does not fail the cell, so
    Run all would carry on past a failed test suite into the real run.
    """
    print("$", *args, flush=True)
    print(flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}; see output above")

## 5. Dependencies

`torch` and `torchvision` are skipped: Colab ships builds matched to its CUDA driver,
and installing over them tends to fail later as a CUDA error inside training rather
than here.

In [ ]:
import subprocess
import sys

PREINSTALLED = {"torch", "torchvision"}

wanted = []
for line in Path("requirements.txt").read_text().splitlines():
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    if line.split("==")[0].split(">=")[0].strip().lower() in PREINSTALLED:
        continue
    wanted.append(line)

# P0.5 only trains. ezkl, streamlit, onnx and cryptography belong to later
# phases and pulling them in here costs minutes of setup for nothing.
NEEDED_FOR_P0_5 = {"numpy", "matplotlib", "pytest"}
wanted = [w for w in wanted if w.split("==")[0].strip().lower() in NEEDED_FOR_P0_5]
print("installing:", wanted)
if wanted:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *wanted], check=True)
print("done")

## 6. Fast checks before spending GPU time

The unit tests, then a 2-epoch run on synthetic data. Together these take about a
minute and catch a broken pipeline before a 30-minute run does. Both raise on failure,
so Run all stops here. If either fails, report the traceback -- do not force section 7.

In [ ]:
run([sys.executable, "-m", "pytest", "tests", "-q"])

In [ ]:
run([sys.executable, "experiments/p0_5_train_clean.py", "--smoke"])

## 7. The real run

One call into `src/`. Checkpoints land in `{DRIVE_ROOT}/checkpoints/p0.5_clean_baseline/`
every epoch, the result JSON in `{DRIVE_ROOT}/results/`, and once all epochs are done the
final weights `W` in `{DRIVE_ROOT}/models/p0.5_clean_baseline_W.pt`.

If this cell dies partway, just run it again -- it resumes from the last epoch. Do not
change EPOCHS, SEED or BATCH_SIZE between attempts: resuming under different
hyperparameters is refused, because it would silently train on the old LR schedule.

In [ ]:
import torch

# Section 3 only prints nvidia-smi. This is the check that actually stops a
# CPU runtime from spending 90 minutes producing an incomplete run.
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then run all again.")

run([
    sys.executable, "experiments/p0_5_train_clean.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
    "--epochs", EPOCHS,
    "--seed", SEED,
    "--batch-size", BATCH_SIZE,
    "--max-minutes", MAX_MINUTES,
    "--num-workers", 2,
])

## 8. Collect what comes back

Prints the numbers for the Results Ledger from the newest result file. Every session that
ends (finished or out of time) writes its own JSON, so there may be several; only the
newest one with `stopped_early: false` is the baseline. If it is true, re-run section 7.

In [ ]:
import json

results = sorted(Path(DRIVE_ROOT, "results").glob("p0.5_clean_baseline__*.json"))
if not results:
    raise FileNotFoundError("no P0.5 result found -- did section 7 finish?")

latest = results[-1]  # filenames end in a UTC timestamp, so this is the newest
record = json.loads(latest.read_text())

print("result file:", latest.name)
print()
m = record["metrics"]
print(f"  epochs completed   : {m['epochs_completed']}/{m['epochs_planned']}")
print(f"  stopped early      : {m['stopped_early']}")
print(f"  TEST ACCURACY      : {m['test_accuracy'] * 100:.2f}%   <-- the P0.6 baseline")
print(f"  best test accuracy : {m['best_test_accuracy'] * 100:.2f}% (epoch {m['best_epoch'] + 1})")
print(f"  holdout accuracy   : {m['holdout_accuracy'] * 100:.2f}%  (sanity check)")
print(f"  training time      : {m['total_epoch_seconds'] / 60:.1f} min, summed over all sessions")
print(f"  gpu                : {record['params']['gpu']}")
print(f"  code commit        : {record['git']['commit']} (source: {record['git']['source']}, dirty: {record['git']['dirty']})")
print(f"  weights (W)        : {m['weights_file']}")
print(f"  weights sha256     : {m['weights_sha256']}")
print(f"  mean epoch seconds : {sum(h['epoch_seconds'] for h in m['history']) / len(m['history']):.1f}")

if m["stopped_early"]:
    print("\n*** INCOMPLETE -- re-run section 7 to continue. Do not report this as final. ***")
else:
    print("\nComplete. Copy the result JSON back into the repo at results/.")

In [ ]:
# Checkpoint inventory, so it is obvious the per-epoch requirement was met.
ckpt_dir = Path(DRIVE_ROOT, "checkpoints", "p0.5_clean_baseline")
for p in [*sorted(ckpt_dir.iterdir()), *sorted(Path(DRIVE_ROOT, "models").glob("p0.5_clean_baseline*"))]:
    print(f"  {p.name:>28}  {p.stat().st_size / 1024**2:.2f} MB")
print("\nmanifest:", (ckpt_dir / "manifest.json").read_text())

## 9. Bring back to the repo

1. The **newest** `{DRIVE_ROOT}/results/p0.5_clean_baseline__*.json` (the one section 8
   printed) → commit into `results/`. Older ones are incomplete sessions.
2. `{DRIVE_ROOT}/models/p0.5_clean_baseline_W.pt` → keep on Drive. This is `W`: the
   final-epoch weights, which later phases load. `*.pt` is gitignored.
   `checkpoints/.../best.pt` is **not** `W`; it was picked by test accuracy.
3. The printed numbers from section 8.

Then report back. The Results Ledger (sections 8.1 and 8.2) and the P0.5 checkbox get
filled in from those files, not from anything printed in this notebook alone.

Finally run the cell below. Drive writes are buffered and synced in the background;
flushing before the runtime goes away is what guarantees the last files reached Drive.

If the run failed, report the traceback instead: per CLAUDE.md 0.1 the checkbox becomes
`[!]` with a note, and nothing gets invented in its place.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
print("Drive flushed and unmounted. Remount (section 1) before running anything else.")